# Day 15 — Damped Holt ve Theta Model Deneyi

Bu notebook'ta mevcut forecasting modellerine alternatif olarak iki klasik
zaman serisi modeli test edilmektedir:

- Damped Holt
- Theta Model

Amaç, daha karmaşık modellere geçmeden önce küçük veri seti ve kısa forecast
horizonuna uygun iki güçlü klasik yöntemin mevcut final modellerden daha iyi
sonuç verip vermediğini incelemektir.

Modeller mevcut değerlendirme sistemiyle aynı şekilde:

- 12 fold
- her fold için 4 haftalık test horizon'u
- MAE ve RMSE
- Google Trends için 0–100 clipping

kullanılarak değerlendirilecektir.

Yeni modeller yalnızca deneysel olarak test edilmektedir. Sonuçlar görülmeden
final forecasting pipeline'ında herhangi bir değişiklik yapılmayacaktır.

In [1]:
# --------------------------------------------------
# Day 15 - Damped Holt ve Theta Model Deneyi
# --------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd

# MAE ve RMSE hesaplamak için kullandığımız
# mevcut evaluation fonksiyonları.
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
)

# Zaman sırasını koruyan cross-validation yapısı.
from sklearn.model_selection import TimeSeriesSplit

# --------------------------------------------------
# Yeni Model 1: Damped Holt
# --------------------------------------------------

# ExponentialSmoothing:
# Level ve trend gibi zaman serisi bileşenlerini
# geçmiş gözlemlere daha fazla ağırlık vererek modeller.
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# --------------------------------------------------
# Yeni Model 2: Theta
# --------------------------------------------------

from statsmodels.tsa.forecasting.theta import ThetaModel


# --------------------------------------------------
# Proje yolu
# --------------------------------------------------

# Notebook notebooks/ klasöründe olduğu için
# .parent ile proje ana klasörüne çıkıyoruz.
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "google_trends_ai_3y_updated_2026-08-09.csv"
)


# --------------------------------------------------
# Güncel veriyi yükleme
# --------------------------------------------------

data = pd.read_csv(
    DATA_PATH,
    parse_dates=["date"],
    index_col="date",
)


# --------------------------------------------------
# Mevcut model karşılaştırmamızla aynı tarih aralığı
# --------------------------------------------------

# XGBoost final modelimiz 8 lag kullandığı için
# model karşılaştırmalarında ilk 8 haftayı dışarıda
# bırakmıştık.
#
# Yeni modelleri de aynı tarihler üzerinde
# değerlendirmeliyiz.
#
# Aksi halde örneğin Holt 159 haftada,
# XGBoost 151 haftada değerlendirilmiş olur ve
# karşılaştırma tam olarak adil olmaz.
N_LAGS = 8

aligned_data = data.iloc[N_LAGS:].copy()


# --------------------------------------------------
# Cross-validation yapısı
# --------------------------------------------------

# n_splits=12:
# 12 farklı geçmiş dönemde modeli test ediyoruz.
#
# test_size=4:
# Her fold'da gelecek 4 haftayı tahmin ediyoruz.
#
# Böylece yaklaşık 30 günlük gerçek proje
# forecast horizonumuzu taklit ediyoruz.
splitter = TimeSeriesSplit(
    n_splits=12,
    test_size=4,
)


# Google Trends değerlerinin doğal aralığı.
CLIP_RANGE = (0, 100)


print("Orijinal veri sayısı:", len(data))
print("Hizalanmış veri sayısı:", len(aligned_data))
print()
print("Başlangıç:", aligned_data.index.min())
print("Bitiş:", aligned_data.index.max())

Orijinal veri sayısı: 159
Hizalanmış veri sayısı: 151

Başlangıç: 2023-09-24 00:00:00
Bitiş: 2026-08-09 00:00:00


In [2]:
def evaluate_damped_holt_cv(
    series: pd.Series,
    splitter: TimeSeriesSplit,
    clip_range: tuple[float, float] | None = None,
) -> pd.DataFrame:
    """
    Damped Holt modelini time-series cross-validation ile değerlendirir.

    Parameters
    ----------
    series : pd.Series
        Değerlendirilecek tarihsel zaman serisi.

    splitter : TimeSeriesSplit
        Train ve test dönemlerini zaman sırasını koruyarak oluşturan
        cross-validation nesnesi.

    clip_range : tuple[float, float] | None, default=None
        Tahminlerin tutulacağı alt ve üst sınır.

    Returns
    -------
    pd.DataFrame
        Her fold için MAE ve RMSE sonuçlarını içeren tablo.
    """

    results = []

    for fold, (train_index, test_index) in enumerate(
        splitter.split(series),
        start=1,
    ):
        # ----------------------------------------------
        # Train ve test verisini ayırma
        # ----------------------------------------------

        train = series.iloc[train_index]
        test = series.iloc[test_index]

        # ----------------------------------------------
        # Damped Holt modeli
        # ----------------------------------------------

        model = ExponentialSmoothing(
            train,

            # add:
            # Trendin Google Trends puanı cinsinden
            # toplamsal olarak modellenmesini istiyoruz.
            trend="add",

            # damped_trend=True:
            # Geçmişte görülen trendin geleceğe aynı hızla
            # sonsuza kadar devam etmesini engeller.
            #
            # Trend ileri gittikçe yavaş yavaş sönümlenir.
            damped_trend=True,

            # Şimdilik ayrıca seasonal component eklemiyoruz.
            #
            # Amacımız önce Damped Holt'un level + trend
            # yapısının tek başına performansını görmek.
            seasonal=None,
        )

        fitted_model = model.fit(
            optimized=True,
        )

        # Test horizonumuz 4 hafta olduğu için
        # test uzunluğu kadar tahmin üretiyoruz.
        prediction = fitted_model.forecast(
            steps=len(test),
        )

        # Google Trends değerlerinin 0-100 domain'ine
        # uygun kalmasını sağlıyoruz.
        if clip_range is not None:
            prediction = np.clip(
                prediction,
                clip_range[0],
                clip_range[1],
            )

        mae = mean_absolute_error(
            test,
            prediction,
        )

        rmse = root_mean_squared_error(
            test,
            prediction,
        )

        results.append(
            {
                "fold": fold,
                "MAE": mae,
                "RMSE": rmse,
            }
        )

    return pd.DataFrame(results)


def evaluate_theta_cv(
    series: pd.Series,
    splitter: TimeSeriesSplit,
    clip_range: tuple[float, float] | None = None,
) -> pd.DataFrame:
    """
    Theta modelini time-series cross-validation ile değerlendirir.

    Parameters
    ----------
    series : pd.Series
        Değerlendirilecek tarihsel zaman serisi.

    splitter : TimeSeriesSplit
        Train ve test dönemlerini zaman sırasını koruyarak oluşturan
        cross-validation nesnesi.

    clip_range : tuple[float, float] | None, default=None
        Tahminlerin tutulacağı alt ve üst sınır.

    Returns
    -------
    pd.DataFrame
        Her fold için MAE ve RMSE sonuçlarını içeren tablo.
    """

    results = []

    for fold, (train_index, test_index) in enumerate(
        splitter.split(series),
        start=1,
    ):
        train = series.iloc[train_index]
        test = series.iloc[test_index]

        # ----------------------------------------------
        # Theta modeli
        # ----------------------------------------------

        # deseasonalize=False:
        #
        # İlk deneyde Theta'ya ayrıca 52 haftalık
        # seasonality varsayımı vermiyoruz.
        #
        # Veri setimiz yaklaşık 3 yıllık olduğu için
        # yalnızca birkaç yıllık döngü bulunuyor.
        #
        # Önce daha sade ve daha az varsayımlı
        # Theta yapısını test ediyoruz.
        model = ThetaModel(
            train,
            deseasonalize=False,
        )

        fitted_model = model.fit()

        prediction = fitted_model.forecast(
            steps=len(test),
        )

        if clip_range is not None:
            prediction = np.clip(
                prediction,
                clip_range[0],
                clip_range[1],
            )

        mae = mean_absolute_error(
            test,
            prediction,
        )

        rmse = root_mean_squared_error(
            test,
            prediction,
        )

        results.append(
            {
                "fold": fold,
                "MAE": mae,
                "RMSE": rmse,
            }
        )

    return pd.DataFrame(results)


# --------------------------------------------------
# Üç trend üzerinde iki modeli test etme
# --------------------------------------------------

new_model_results = []

for trend in [
    "chatgpt",
    "gemini",
    "claude",
]:
    series = aligned_data[trend]

    # Damped Holt
    holt_results = evaluate_damped_holt_cv(
        series=series,
        splitter=splitter,
        clip_range=CLIP_RANGE,
    )

    # Theta
    theta_results = evaluate_theta_cv(
        series=series,
        splitter=splitter,
        clip_range=CLIP_RANGE,
    )

    # Her modelin 12 fold'daki ortalama
    # MAE ve RMSE değerlerini kaydediyoruz.
    new_model_results.append(
        {
            "Trend": trend.capitalize(),
            "Model": "Damped Holt",
            "Mean_MAE": holt_results["MAE"].mean(),
            "Mean_RMSE": holt_results["RMSE"].mean(),
        }
    )

    new_model_results.append(
        {
            "Trend": trend.capitalize(),
            "Model": "Theta",
            "Mean_MAE": theta_results["MAE"].mean(),
            "Mean_RMSE": theta_results["RMSE"].mean(),
        }
    )


new_models_df = pd.DataFrame(new_model_results)

new_models_df

/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init

,Trend,Model,Mean_MAE,Mean_RMSE
0,Chatgpt,Damped Holt,5.351379,6.220543
1,Chatgpt,Theta,5.309422,6.182738
2,Gemini,Damped Holt,5.927773,6.759292
3,Gemini,Theta,4.329817,4.908052
4,Claude,Damped Holt,1.805700,1.951513
5,Claude,Theta,1.615661,1.750894


Aaa burada ilginç bir sonuç çıktı 🙂 Özellikle Gemini tarafı beklediğimden daha yakın.

Karşılaştıralım:

| Trend   | Model       |        MAE |   RMSE |   Mevcut final MAE |
| ------- | ----------- | ---------: | -----: | -----------------: |
| ChatGPT | Damped Holt |      5.351 |  6.221 | **3.850 Ensemble** |
| ChatGPT | Theta       |      5.309 |  6.183 | **3.850 Ensemble** |
| Gemini  | Damped Holt |      5.928 |  6.759 |    **4.333 Naive** |
| Gemini  | **Theta**   | **4.3298** | 4.9081 |   **4.3333 Naive** |
| Claude  | Damped Holt |      1.806 |  1.952 |    **1.583 Naive** |
| Claude  | Theta       |      1.616 |  1.751 |    **1.583 Naive** |


Burada net:

Ensemble ≈ 3.850
Theta    ≈ 5.309
Holt     ≈ 5.351

İki yeni model de belirgin şekilde daha kötü. Dolayısıyla ChatGPT için mevcut Prophet + XGBoost Ensemble seçimimiz güçlenmiş oldu.

Claude

Burada da:

Naive ≈ 1.583
Theta ≈ 1.616
Holt  ≈ 1.806

Theta yakın ama Naive hâlâ daha iyi. Üstelik Naive çok daha basit. Claude'u değiştirmek için bir gerekçe yok.

Ama Gemini çok ilginç

Şuna bak:

Naive MAE = 4.333333
Theta MAE = 4.329817

Fark:

4.333333 - 4.329817
≈ 0.0035 puan

Yani MAE'ye göre Theta teknik olarak Naive'ı geçti.

Ama sadece:

0.0035 Google Trends puanı.

Pratik olarak bu fark yok denecek kadar küçük.

Dahası RMSE:

Naive RMSE ≈ 4.9073
Theta RMSE ≈ 4.9081

Burada da bu kez Naive çok küçük farkla daha iyi.

Yani Gemini'de tablo şu:

             MAE       RMSE
Naive       4.3333     4.9073
Theta       4.3298     4.9081


MAE    → Theta minicik daha iyi
RMSE   → Naive minicik daha iyi

Bu sonuç bana şunu söylüyor:

Theta ile Naive'ın Gemini üzerindeki performansı pratik olarak eşit.

Ve böyle bir durumda ben final modeli değiştirmezdim.

Çünkü Naive:

daha basit,
ekstra fitting gerektirmiyor,
çok kolay açıklanıyor,
mevcut pipeline'a zaten entegre,
MAE farkı yalnızca ~0.0035,
RMSE'de de zaten çok küçük farkla önde.

Bu tam bir “daha karmaşık model ölçülebilir bir fayda sağlamıyorsa basit modeli tut” örneği.

Ama Gemini'yi bir kez daha inceleyelim

Burada küçük bir analiz yapmaya değer. Çünkü Theta'nın Naive'ı nasıl yakaladığını görmek güzel olur.

In [3]:
# --------------------------------------------------
# Gemini - Naive vs Theta fold bazlı karşılaştırma
# --------------------------------------------------

gemini_series = aligned_data["gemini"]


# --------------------------------------------------
# Naive sonuçlarını yeniden oluşturma
# --------------------------------------------------

naive_rows = []

for fold, (train_index, test_index) in enumerate(
    splitter.split(gemini_series),
    start=1,
):
    train = gemini_series.iloc[train_index]
    test = gemini_series.iloc[test_index]

    # Naive model:
    # Train dönemindeki son değeri bütün
    # 4 haftalık test horizonuna taşıyoruz.
    prediction = pd.Series(
        train.iloc[-1],
        index=test.index,
    )

    mae = mean_absolute_error(
        test,
        prediction,
    )

    rmse = root_mean_squared_error(
        test,
        prediction,
    )

    naive_rows.append(
        {
            "fold": fold,
            "MAE": mae,
            "RMSE": rmse,
        }
    )


gemini_naive_results = pd.DataFrame(naive_rows)


# --------------------------------------------------
# Theta sonuçlarını yeniden oluşturma
# --------------------------------------------------

gemini_theta_results = evaluate_theta_cv(
    series=gemini_series,
    splitter=splitter,
    clip_range=CLIP_RANGE,
)


# --------------------------------------------------
# Fold bazlı karşılaştırma
# --------------------------------------------------

comparison = pd.DataFrame(
    {
        "Fold": gemini_naive_results["fold"],
        "Naive_MAE": gemini_naive_results["MAE"],
        "Theta_MAE": gemini_theta_results["MAE"],
    }
)


# Her fold'da hangi model daha düşük hata verdi?
comparison["Winner"] = np.where(
    comparison["Theta_MAE"] < comparison["Naive_MAE"],
    "Theta",
    "Naive",
)


comparison

/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/forecasting/theta.py:44: UserWarning: Only PeriodIndexes, DatetimeIndexes with a frequency set, RangesIndexes, and Index with a unit increment support extending. The index is set will contain the position relative to the data length.
  return DeterministicTerm._extend_index(index, steps)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/forecasting/theta.py:44: UserWarning: Only PeriodIndexes, DatetimeIndexes with a frequency set, RangesIndexes, and Index with a unit increment support extending. The index is set will contain the position relative to the data length.
  return DeterministicTerm._extend_index(index, steps)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/forecasting/theta.py:44: UserWarning: Only PeriodIndexes, DatetimeIndexes with a frequency set, RangesIndex

,Fold,Naive_MAE,Theta_MAE,Winner
0,1,21.00,21.136906,Naive
1,2,5.00,5.291927,Naive
2,3,2.50,2.183514,Theta
3,4,1.75,2.088378,Naive
4,5,4.75,4.400852,Theta
5,6,1.50,1.572051,Naive
6,7,4.00,3.633123,Theta
7,8,1.75,2.131931,Naive
8,9,3.25,2.856943,Theta
9,10,1.75,1.912344,Naive


In [4]:
print(
    "Naive Mean MAE:",
    comparison["Naive_MAE"].mean(),
)

print(
    "Theta Mean MAE:",
    comparison["Theta_MAE"].mean(),
)

print()

print(
    "Naive Median MAE:",
    comparison["Naive_MAE"].median(),
)

print(
    "Theta Median MAE:",
    comparison["Theta_MAE"].median(),
)

print()

print("Fold kazananları:")

print(
    comparison["Winner"].value_counts()
)

Naive Mean MAE: 4.333333333333333
Theta Mean MAE: 4.329816901503061

Naive Median MAE: 2.625
Theta Median MAE: 2.520228132133106

Fold kazananları:
Winner
Naive    7
Theta    5
Name: count, dtype: int64


## State-Space Local Linear Trend modeli

In [5]:
from statsmodels.tsa.statespace.structural import UnobservedComponents

In [6]:
def evaluate_local_linear_trend_cv(
    series: pd.Series,
    splitter: TimeSeriesSplit,
    clip_range: tuple[float, float] | None = None,
) -> pd.DataFrame:
    """
    Local Linear Trend state-space modelini
    time-series cross-validation ile değerlendirir.

    Parameters
    ----------
    series : pd.Series
        Değerlendirilecek tarihsel zaman serisi.

    splitter : TimeSeriesSplit
        Zaman sırasını koruyan cross-validation yapısı.

    clip_range : tuple[float, float] | None, default=None
        Tahminlerin tutulacağı alt ve üst sınır.

    Returns
    -------
    pd.DataFrame
        Her fold için MAE ve RMSE sonuçlarını içeren tablo.
    """

    results = []

    for fold, (train_index, test_index) in enumerate(
        splitter.split(series),
        start=1,
    ):
        # Her fold için geçmiş veriyi train,
        # sonraki 4 haftayı test olarak ayırıyoruz.
        train = series.iloc[train_index]
        test = series.iloc[test_index]

        # --------------------------------------------------
        # Local Linear Trend modeli
        # --------------------------------------------------

        # Bu model zaman serisini kabaca iki hareketli
        # parçayla düşünür:
        #
        # level -> serinin mevcut seviyesi
        # trend -> seviyenin hangi yönde/hızda değiştiği
        #
        # "local" olması önemli:
        # level ve trend bütün veri boyunca sabit olmak
        # zorunda değildir; zaman içerisinde değişebilir.
        model = UnobservedComponents(
            train,
            level="local linear trend",
        )

        # Model parametreleri train verisinden tahmin edilir.
        fitted_model = model.fit(
            disp=False,
        )

        # Test horizonumuz kadar ileri tahmin üretiyoruz.
        prediction = fitted_model.forecast(
            steps=len(test),
        )

        # Google Trends'in doğal 0-100 aralığını koruyoruz.
        if clip_range is not None:
            prediction = np.clip(
                prediction,
                clip_range[0],
                clip_range[1],
            )

        # Fold performansı.
        mae = mean_absolute_error(
            test,
            prediction,
        )

        rmse = root_mean_squared_error(
            test,
            prediction,
        )

        results.append(
            {
                "fold": fold,
                "MAE": mae,
                "RMSE": rmse,
            }
        )

    return pd.DataFrame(results)

In [7]:
local_trend_summary = []

for trend in [
    "chatgpt",
    "gemini",
    "claude",
]:
    series = aligned_data[trend]

    results = evaluate_local_linear_trend_cv(
        series=series,
        splitter=splitter,
        clip_range=CLIP_RANGE,
    )

    local_trend_summary.append(
        {
            "Trend": trend.capitalize(),
            "Model": "Local Linear Trend",
            "Mean_MAE": results["MAE"].mean(),
            "Mean_RMSE": results["RMSE"].mean(),
        }
    )


local_trend_df = pd.DataFrame(local_trend_summary)

local_trend_df

/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init

,Trend,Model,Mean_MAE,Mean_RMSE
0,Chatgpt,Local Linear Trend,5.588239,6.467505
1,Gemini,Local Linear Trend,5.933988,6.789670
2,Claude,Local Linear Trend,1.861052,2.015149


## SARIMAX / Seasonal ARIMA Deneyi

Bu bölümde haftalık Google Trends serilerinde yaklaşık yıllık bir seasonal
yapının forecasting performansına katkı sağlayıp sağlamadığı incelenmektedir.

Haftalık veri kullanıldığı için seasonal period:

`52 hafta`

olarak ele alınmaktadır.

Veri setinde yaklaşık 3 yıllık gözlem bulunduğu için yıllık seasonality
konusunda güçlü bir varsayım yapılmaması adına yalnızca birkaç basit
SARIMAX konfigürasyonu test edilmektedir.

Bu deneyde exogenous variable kullanılmamaktadır. Amaç SARIMAX modelinin
seasonal AR ve seasonal MA bileşenlerinin mevcut modellere göre bir avantaj
sağlayıp sağlamadığını incelemektir.

In [9]:
# SARIMAX:
# Seasonal ARIMA ve gerektiğinde exogenous variable
# kullanımını destekleyen statsmodels modelidir.
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [10]:
def evaluate_sarimax_cv(
    series: pd.Series,
    splitter: TimeSeriesSplit,
    order: tuple[int, int, int],
    seasonal_order: tuple[int, int, int, int],
    clip_range: tuple[float, float] | None = None,
) -> pd.DataFrame:
    """
    SARIMAX modelini time-series cross-validation ile değerlendirir.

    Parameters
    ----------
    series : pd.Series
        Değerlendirilecek tarihsel zaman serisi.

    splitter : TimeSeriesSplit
        Zaman sırasını koruyarak train ve test fold'larını oluşturan
        cross-validation nesnesi.

    order : tuple[int, int, int]
        Modelin normal ARIMA kısmındaki (p, d, q) parametreleri.

    seasonal_order : tuple[int, int, int, int]
        Modelin seasonal kısmındaki (P, D, Q, s) parametreleri.

    clip_range : tuple[float, float] | None, default=None
        Tahminlerin tutulacağı alt ve üst sınır.

    Returns
    -------
    pd.DataFrame
        Her fold için MAE ve RMSE sonuçlarını içeren tablo.
    """

    results = []

    for fold, (train_index, test_index) in enumerate(
        splitter.split(series),
        start=1,
    ):
        # --------------------------------------------------
        # Train / test ayrımı
        # --------------------------------------------------

        train = series.iloc[train_index]
        test = series.iloc[test_index]

        # --------------------------------------------------
        # SARIMAX modeli
        # --------------------------------------------------

        model = SARIMAX(
            train,

            # Normal ARIMA kısmı:
            # örneğin (1, 1, 1).
            order=order,

            # Seasonal ARIMA kısmı:
            #
            # (P, D, Q, s)
            #
            # s=52:
            # haftalık veride yaklaşık 1 yıllık seasonal period.
            seasonal_order=seasonal_order,

            # Bu iki parametre modelin bazı matematiksel
            # kısıtlarını daha esnek hale getirir.
            #
            # Özellikle küçük veri setlerinde model fitting
            # sırasında sorun yaşama ihtimalini azaltabilir.
            enforce_stationarity=False,
            enforce_invertibility=False,
        )

        # disp=False:
        # optimizasyon sırasında terminale uzun çıktı basılmasını
        # engeller.
        fitted_model = model.fit(
            disp=False,
        )

        # Her fold'daki 4 haftalık test horizon'u
        # kadar ileri tahmin oluşturuyoruz.
        prediction = fitted_model.forecast(
            steps=len(test),
        )

        # --------------------------------------------------
        # Google Trends domain kontrolü
        # --------------------------------------------------

        if clip_range is not None:
            prediction = np.clip(
                prediction,
                clip_range[0],
                clip_range[1],
            )

        # --------------------------------------------------
        # Evaluation
        # --------------------------------------------------

        mae = mean_absolute_error(
            test,
            prediction,
        )

        rmse = root_mean_squared_error(
            test,
            prediction,
        )

        results.append(
            {
                "fold": fold,
                "MAE": mae,
                "RMSE": rmse,
            }
        )

    return pd.DataFrame(results)

In [11]:
# --------------------------------------------------
# Denenecek SARIMAX konfigürasyonları
# --------------------------------------------------

sarimax_configs = {
    "SARIMAX_AR52": {
        "order": (1, 1, 1),
        "seasonal_order": (1, 0, 0, 52),
    },
    "SARIMAX_MA52": {
        "order": (1, 1, 1),
        "seasonal_order": (0, 0, 1, 52),
    },
}


# Bütün trend + model kombinasyonlarının
# özet sonuçlarını burada tutacağız.
sarimax_summary = []


for trend in [
    "chatgpt",
    "gemini",
    "claude",
]:
    # Daha önce diğer modellerde kullandığımız
    # aynı hizalanmış seriyi kullanıyoruz.
    series = aligned_data[trend]

    # Dictionary içerisindeki iki SARIMAX
    # konfigürasyonunu sırayla deniyoruz.
    for model_name, config in sarimax_configs.items():

        print(
            f"Çalışıyor: {trend} - {model_name}"
        )

        results = evaluate_sarimax_cv(
            series=series,
            splitter=splitter,
            order=config["order"],
            seasonal_order=config["seasonal_order"],
            clip_range=CLIP_RANGE,
        )

        sarimax_summary.append(
            {
                "Trend": trend.capitalize(),
                "Model": model_name,
                "Mean_MAE": results["MAE"].mean(),
                "Mean_RMSE": results["RMSE"].mean(),
            }
        )


sarimax_df = pd.DataFrame(sarimax_summary)

sarimax_df

Çalışıyor: chatgpt - SARIMAX_AR52


/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init

Çalışıyor: chatgpt - SARIMAX_MA52


/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequ

Çalışıyor: gemini - SARIMAX_AR52


...

Çalışıyor: claude - SARIMAX_AR52


/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init

Çalışıyor: claude - SARIMAX_MA52


/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequ

,Trend,Model,Mean_MAE,Mean_RMSE
0,Chatgpt,SARIMAX_AR52,5.370000,6.172021
1,Chatgpt,SARIMAX_MA52,4.818838,5.680961
2,Gemini,SARIMAX_AR52,5.632760,6.492360
3,Gemini,SARIMAX_MA52,5.525478,6.354315
4,Claude,SARIMAX_AR52,2.016442,2.181073
5,Claude,SARIMAX_MA52,2.043980,2.212964


### SARIMAX Deneyi Sonucu

Haftalık veride yaklaşık yıllık seasonality etkisini incelemek amacıyla
52 haftalık seasonal period kullanan iki SARIMAX konfigürasyonu test edildi:

- SARIMAX(1,1,1) × (1,0,0,52)
- SARIMAX(1,1,1) × (0,0,1,52)

ChatGPT serisinde seasonal MA yapısı seasonal AR yapısından daha düşük hata
vermesine rağmen mevcut Prophet + XGBoost Ensemble modelini geçemedi.

Gemini ve Claude serilerinde de SARIMAX modelleri mevcut Naive modellerden
daha yüksek hata verdi.

Yaklaşık üç yıllık haftalık veri içerisinde yalnızca birkaç yıllık seasonal
döngü bulunduğu için 52 haftalık güçlü bir seasonality varsayımının forecasting
performansına yeterli katkı sağlamadığı değerlendirildi.

Bu nedenle SARIMAX final forecasting pipeline'ına dahil edilmedi.